# Inferência com o LoRA treinado — EscutIA

Este notebook carrega o modelo base `Qwen/Qwen2.5-0.5B-Instruct` junto com o adapter LoRA produzido pelo notebook `Fine_Tuning_LoRA_EscutIA.ipynb`.

Antes de executar as células de inferência, o treinamento precisa ter terminado e o diretório `outputs/resultados/lora_escutia` precisa conter `adapter_config.json` e os pesos do adapter.

## 1. Preparar caminhos e dispositivo

A célula funciona tanto quando o notebook é aberto dentro de `EscutIA/fine_tuning_lora` quanto quando o Jupyter é iniciado na raiz do projeto.

In [1]:
from pathlib import Path
import re
import torch
from IPython.display import display

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'configs').exists() and (BASE_DIR / 'EscutIA' / 'fine_tuning_lora' / 'configs').exists():
    BASE_DIR = BASE_DIR / 'EscutIA' / 'fine_tuning_lora'

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
ADAPTER_DIR = BASE_DIR / 'outputs' / 'resultados' / 'lora_escutia'

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f'Adapter não encontrado em {ADAPTER_DIR}. Execute primeiro o fine-tuning e aguarde o término.'
    )
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise FileNotFoundError(
        f'{ADAPTER_DIR} existe, mas não contém adapter_config.json. '
        'Verifique se o treinamento LoRA foi concluído e se o output_dir da configuração está correto.'
    )

if getattr(torch, 'cuda', None) is not None and torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif getattr(torch, 'xpu', None) is not None and torch.xpu.is_available():
    DEVICE = torch.device('xpu')
    DTYPE = torch.bfloat16
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE = torch.float32
else:
    DEVICE = torch.device('cpu')
    DTYPE = torch.float32

print(f'Diretório base: {BASE_DIR}')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Dispositivo: {DEVICE} | dtype: {DTYPE}')

Diretório base: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora
Adapter: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora\outputs\resultados\lora_escutia
Dispositivo: xpu | dtype: torch.bfloat16


## 2. Carregar o modelo base e o adapter LoRA

O adapter não é um modelo completo: ele precisa ser aplicado sobre o mesmo modelo base usado no treinamento.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=DTYPE,
)
modelo = PeftModel.from_pretrained(modelo_base, ADAPTER_DIR)
modelo = modelo.to(DEVICE)
modelo.eval()

print('Modelo base + adapter carregados com sucesso.')
print(f'Parâmetros treináveis restantes: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}')

`torch_dtype` is deprecated! Use `dtype` instead!


Modelo base + adapter carregados com sucesso.
Parâmetros treináveis restantes: 0


## 3. Criar a função de classificação

O formato da mensagem replica o formato conversacional preparado para o dataset: instrução de classificação, texto do usuário e geração do rótulo. A decodificação retorna tanto a resposta bruta quanto o rótulo identificado.

In [5]:
INSTRUCTION = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo.'
LABELS = ('negativo', 'neutro', 'positivo')

def _extrair_rotulo(resposta: str):
    resposta_normalizada = resposta.casefold()
    encontrados = [
        rotulo for rotulo in LABELS
        if re.search(rf'\b{re.escape(rotulo)}\b', resposta_normalizada)
    ]
    return encontrados[0] if encontrados else None

def classificar_sentimento(texto: str, max_new_tokens: int = 8):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError('Informe um texto não vazio.')

    mensagens = [
        {'role': 'system', 'content': 'Você classifica sentimentos em textos em português.'},
        {'role': 'user', 'content': f'{INSTRUCTION}\n\nTexto: {texto.strip()}'},
    ]
    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )
    entradas = tokenizer(prompt, return_tensors='pt')
    entradas = {nome: valor.to(DEVICE) for nome, valor in entradas.items()}

    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    novos_tokens = saida[0, entradas['input_ids'].shape[1]:]
    resposta = tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
    return {'texto': texto, 'resposta_bruta': resposta, 'rotulo': _extrair_rotulo(resposta)}

## 4. Testar exemplos

Execute esta célula para observar as respostas do adapter treinado.

In [7]:
exemplos = [
    'Estou muito feliz com a solução que encontrei para o problema.',
    'A reunião aconteceu conforme o planejado.',
    'Estou preocupado e frustrado com o resultado da prova.',
    'Hoje recebi uma notícia inesperada: preciso pensar melhor antes de responder.',
]

resultados = [classificar_sentimento(texto) for texto in exemplos]
display(__import__('pandas').DataFrame(resultados))

,texto,resposta_bruta,rotulo
0,Estou muito feliz com a solução que encontrei ...,positivo,positivo
1,A reunião aconteceu conforme o planejado.,positivo,positivo
2,Estou preocupado e frustrado com o resultado d...,negativo,negativo
3,Hoje recebi uma notícia inesperada: preciso pe...,negativo,negativo


## 5. Testar um texto próprio

Edite o texto abaixo e execute a célula. O campo `resposta_bruta` ajuda a identificar quando o modelo gerou algo diferente de um dos três rótulos esperados.

In [8]:
texto_teste = 'Escreva aqui o texto que você quer classificar.'
resultado = classificar_sentimento(texto_teste)
print('Texto:', resultado['texto'])
print('Resposta do modelo:', resultado['resposta_bruta'])
print('Rótulo identificado:', resultado['rotulo'] or 'não identificado')

Texto: Escreva aqui o texto que você quer classificar.
Resposta do modelo: neutro
Rótulo identificado: neutro
